## XGBoost Model for Expected Attention

In [9]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [10]:
# load X_train and y_train
X_train = pd.read_csv("../outputs_csv/X_train.csv")
y_train = pd.read_csv("../outputs_csv/y_train.csv")
y_train = y_train["avg_attention_score"]

In [11]:
# implement cross validation using an 80-20 split.
# 5 folds means each validation fold is 20% of the data and each training fold is 80%.
N_SPLITS = 5
RANDOM_STATE = 42

if len(X_train) != len(y_train):
    raise ValueError(
        f"X_train and y_train must have the same number of rows. "
        f"Got {len(X_train)} and {len(y_train)}."
    )

kfold = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

cv_folds = []
for fold_number, (train_idx, val_idx) in enumerate(kfold.split(X_train), start=1):
    cv_folds.append(
        {
            "fold": fold_number,
            "train_idx": train_idx,
            "val_idx": val_idx,
            "X_train_fold": X_train.iloc[train_idx],
            "X_val_fold": X_train.iloc[val_idx],
            "y_train_fold": y_train.iloc[train_idx],
            "y_val_fold": y_train.iloc[val_idx],
        }
    )

cv_summary = pd.DataFrame(
    [
        {
            "fold": fold["fold"],
            "train_rows": len(fold["train_idx"]),
            "validation_rows": len(fold["val_idx"]),
            "validation_pct": len(fold["val_idx"]) / len(X_train),
        }
        for fold in cv_folds
    ]
)

cv_summary

,fold,train_rows,validation_rows,validation_pct
0,1,24776,6195,0.200026
1,2,24777,6194,0.199994
2,3,24777,6194,0.199994
3,4,24777,6194,0.199994
4,5,24777,6194,0.199994


### Creating the Model

In [12]:
try:
    from xgboost import XGBRegressor
except ImportError as exc:
    raise ImportError(
        "xgboost is required for the model cells below. "
        "Install it in this notebook kernel before running these cells."
    ) from exc

N_MONTE_CARLO_SIMULATIONS = 10
MONTE_CARLO_SEEDS = [RANDOM_STATE + seed_offset for seed_offset in range(N_MONTE_CARLO_SIMULATIONS)]


def calculate_regression_metrics(y_true, y_pred):
    return {
        "rmse": np.sqrt(mean_squared_error(y_true, y_pred)),
        "mae": mean_absolute_error(y_true, y_pred),
        "r2": r2_score(y_true, y_pred),
    }


def evaluate_xgboost_monte_carlo(model_name, model_params, cv_folds, monte_carlo_seeds):
    fold_metrics = []

    for fold in cv_folds:
        simulation_predictions = []

        for seed in monte_carlo_seeds:
            model = XGBRegressor(**model_params, random_state=seed)
            model.fit(fold["X_train_fold"], fold["y_train_fold"])
            simulation_predictions.append(model.predict(fold["X_val_fold"]))

        averaged_predictions = np.mean(simulation_predictions, axis=0)
        metrics = calculate_regression_metrics(fold["y_val_fold"], averaged_predictions)
        metrics.update(
            {
                "model": model_name,
                "fold": fold["fold"],
                "num_simulations": len(monte_carlo_seeds),
                "mean_prediction_std": np.mean(np.std(simulation_predictions, axis=0)),
            }
        )
        fold_metrics.append(metrics)

    fold_metrics_df = pd.DataFrame(fold_metrics)
    summary = pd.DataFrame(
        [
            {
                "model": model_name,
                "num_simulations": len(monte_carlo_seeds),
                "mean_rmse": fold_metrics_df["rmse"].mean(),
                "std_rmse": fold_metrics_df["rmse"].std(),
                "mean_mae": fold_metrics_df["mae"].mean(),
                "std_mae": fold_metrics_df["mae"].std(),
                "mean_r2": fold_metrics_df["r2"].mean(),
                "std_r2": fold_metrics_df["r2"].std(),
                "mean_prediction_std": fold_metrics_df["mean_prediction_std"].mean(),
            }
        ]
    )

    return fold_metrics_df, summary

### Low-Variance XGBoost

In [13]:
low_variance_params = {
    "objective": "reg:squarederror",
    "eval_metric": "rmse",
    "n_estimators": 200,
    "learning_rate": 0.04,
    "max_depth": 2,
    "min_child_weight": 8,
    "gamma": 0.2,
    "subsample": 0.80,
    "colsample_bytree": 0.80,
    "reg_alpha": 0.10,
    "reg_lambda": 5.0,
    "tree_method": "hist",
    "n_jobs": -1,
    "verbosity": 0,
}

low_variance_fold_metrics, low_variance_summary = evaluate_xgboost_monte_carlo(
    model_name="low_variance_xgboost",
    model_params=low_variance_params,
    cv_folds=cv_folds,
    monte_carlo_seeds=MONTE_CARLO_SEEDS,
)

display(low_variance_fold_metrics)
low_variance_summary

,rmse,mae,r2,model,fold,num_simulations,mean_prediction_std
0,0.379941,0.291896,0.397578,low_variance_xgboost,1,10,0.004810
1,0.385138,0.299182,0.403800,low_variance_xgboost,2,10,0.004911
2,0.380147,0.293480,0.395226,low_variance_xgboost,3,10,0.004931
3,0.381226,0.293145,0.399081,low_variance_xgboost,4,10,0.004769
4,0.386563,0.296712,0.390587,low_variance_xgboost,5,10,0.004717


,model,num_simulations,mean_rmse,std_rmse,mean_mae,std_mae,mean_r2,std_r2,mean_prediction_std
0,low_variance_xgboost,10,0.382603,0.003046,0.294883,0.002989,0.397254,0.004869,0.004828


### Medium-Variance XGBoost

In [14]:
medium_variance_params = {
    "objective": "reg:squarederror",
    "eval_metric": "rmse",
    "n_estimators": 350,
    "learning_rate": 0.05,
    "max_depth": 4,
    "min_child_weight": 4,
    "gamma": 0.05,
    "subsample": 0.90,
    "colsample_bytree": 0.90,
    "reg_alpha": 0.02,
    "reg_lambda": 2.0,
    "tree_method": "hist",
    "n_jobs": -1,
    "verbosity": 0,
}

medium_variance_fold_metrics, medium_variance_summary = evaluate_xgboost_monte_carlo(
    model_name="medium_variance_xgboost",
    model_params=medium_variance_params,
    cv_folds=cv_folds,
    monte_carlo_seeds=MONTE_CARLO_SEEDS,
)

display(medium_variance_fold_metrics)
medium_variance_summary

,rmse,mae,r2,model,fold,num_simulations,mean_prediction_std
0,0.378021,0.290725,0.403651,medium_variance_xgboost,1,10,0.011774
1,0.384038,0.298089,0.407202,medium_variance_xgboost,2,10,0.011715
2,0.378303,0.291897,0.401080,medium_variance_xgboost,3,10,0.011750
3,0.379726,0.292305,0.403799,medium_variance_xgboost,4,10,0.011959
4,0.386175,0.296376,0.391808,medium_variance_xgboost,5,10,0.011742


,model,num_simulations,mean_rmse,std_rmse,mean_mae,std_mae,mean_r2,std_r2,mean_prediction_std
0,medium_variance_xgboost,10,0.381253,0.003656,0.293878,0.003175,0.401508,0.005842,0.011788


### High-Variance XGBoost

In [15]:
high_variance_params = {
    "objective": "reg:squarederror",
    "eval_metric": "rmse",
    "n_estimators": 500,
    "learning_rate": 0.06,
    "max_depth": 7,
    "min_child_weight": 1,
    "gamma": 0.0,
    "subsample": 0.95,
    "colsample_bytree": 0.95,
    "reg_alpha": 0.0,
    "reg_lambda": 0.5,
    "tree_method": "hist",
    "n_jobs": -1,
    "verbosity": 0,
}

high_variance_fold_metrics, high_variance_summary = evaluate_xgboost_monte_carlo(
    model_name="high_variance_xgboost",
    model_params=high_variance_params,
    cv_folds=cv_folds,
    monte_carlo_seeds=MONTE_CARLO_SEEDS,
)

display(high_variance_fold_metrics)
high_variance_summary

,rmse,mae,r2,model,fold,num_simulations,mean_prediction_std
0,0.388981,0.300443,0.368569,high_variance_xgboost,1,10,0.038154
1,0.393556,0.306010,0.377452,high_variance_xgboost,2,10,0.037839
2,0.387107,0.298484,0.372879,high_variance_xgboost,3,10,0.038003
3,0.390737,0.300895,0.368723,high_variance_xgboost,4,10,0.037859
4,0.397218,0.304821,0.356528,high_variance_xgboost,5,10,0.037729


,model,num_simulations,mean_rmse,std_rmse,mean_mae,std_mae,mean_r2,std_r2,mean_prediction_std
0,high_variance_xgboost,10,0.39152,0.003973,0.302131,0.003161,0.36883,0.007781,0.037917


### Model Comparison

In [16]:
model_comparison = (
    pd.concat(
        [low_variance_summary, medium_variance_summary, high_variance_summary],
        ignore_index=True,
    )
    .sort_values(["mean_rmse", "mean_mae", "mean_r2"], ascending=[True, True, False])
    .reset_index(drop=True)
)

best_model_name = model_comparison.loc[0, "model"]
print(f"Best model by mean validation RMSE: {best_model_name}")

model_comparison

Best model by mean validation RMSE: medium_variance_xgboost


,model,num_simulations,mean_rmse,std_rmse,mean_mae,std_mae,mean_r2,std_r2,mean_prediction_std
0,medium_variance_xgboost,10,0.381253,0.003656,0.293878,0.003175,0.401508,0.005842,0.011788
1,low_variance_xgboost,10,0.382603,0.003046,0.294883,0.002989,0.397254,0.004869,0.004828
2,high_variance_xgboost,10,0.391520,0.003973,0.302131,0.003161,0.368830,0.007781,0.037917


In [17]:
# Use the medium-variance XGBoost profile to generate out-of-fold expected attention.
# Gravity = actual attention - expected attention.
def predict_expected_attention_monte_carlo_cv(model_params, cv_folds, monte_carlo_seeds, num_rows):
    expected_attention = np.full(num_rows, np.nan)

    for fold in cv_folds:
        simulation_predictions = []

        for seed in monte_carlo_seeds:
            model = XGBRegressor(**model_params, random_state=seed)
            model.fit(fold["X_train_fold"], fold["y_train_fold"])
            simulation_predictions.append(model.predict(fold["X_val_fold"]))

        expected_attention[fold["val_idx"]] = np.mean(simulation_predictions, axis=0)

    if np.isnan(expected_attention).any():
        raise ValueError("Some rows did not receive expected_attention predictions.")

    return expected_attention


y_train_with_players = pd.read_csv("../outputs_csv/y_train.csv")
gravity_dataset = y_train_with_players.copy()
gravity_dataset["expected_attention"] = predict_expected_attention_monte_carlo_cv(
    model_params=medium_variance_params,
    cv_folds=cv_folds,
    monte_carlo_seeds=MONTE_CARLO_SEEDS,
    num_rows=len(X_train),
)
gravity_dataset["gravity_score"] = (
    gravity_dataset["avg_attention_score"] - gravity_dataset["expected_attention"]
)

gravity_dataset.head()

,avg_attention_score,rusher_name,expected_attention,gravity_score
0,0.461538,Demarcus Lawrence,0.904706,-0.443167
1,0.615385,Randy Gregory,0.368581,0.246803
2,1.307692,Carlos Watkins,1.218276,0.089416
3,1.000000,Micah Parsons,1.141261,-0.141261
4,0.615385,Osa Odighizuwa,1.118670,-0.503285


### Gravity Scores for Selected Players

In [18]:
selected_players = ["Myles Garrett", "Aaron Donald", "Justin Hollins"]

selected_player_gravity = (
    gravity_dataset[gravity_dataset["rusher_name"].isin(selected_players)]
    .groupby("rusher_name", as_index=False)
    .agg(
        num_rows=("gravity_score", "size"),
        actual_attention=("avg_attention_score", "mean"),
        expected_attention=("expected_attention", "mean"),
        gravity_score=("gravity_score", "mean"),
    )
    .sort_values("gravity_score", ascending=False)
    .reset_index(drop=True)
)

selected_player_gravity

,rusher_name,num_rows,actual_attention,expected_attention,gravity_score
0,Aaron Donald,229,1.590982,1.362093,0.228889
1,Myles Garrett,170,1.084141,0.978004,0.106137
2,Justin Hollins,64,0.728201,0.836464,-0.108263


### Top 20 Gravity Scores

In [24]:
top_20_gravity_scores = (
    gravity_dataset
    .groupby("rusher_name", as_index=False)
    .agg(
        num_rows=("gravity_score", "size"),
        actual_attention=("avg_attention_score", "mean"),
        expected_attention=("expected_attention", "mean"),
        gravity_score=("gravity_score", "mean"),
    )
    .query("num_rows >= 75")
    .sort_values("gravity_score", ascending=False)
    .head(20)
    .reset_index(drop=True)
)

top_20_gravity_scores

,rusher_name,num_rows,actual_attention,expected_attention,gravity_score
0,Aaron Donald,229,1.590982,1.362093,0.228889
1,Christian Barmore,165,1.612486,1.456215,0.156271
2,Cameron Jordan,197,1.122000,0.993966,0.128034
3,Matt Ioannidis,122,1.522077,1.407333,0.114744
4,Michael Brockers,108,1.508740,1.396328,0.112412
5,Nick Bosa,138,1.059147,0.952817,0.106330
6,Myles Garrett,170,1.084141,0.978004,0.106137
7,Grady Jarrett,159,1.541059,1.446504,0.094555
8,Payton Turner,77,1.202520,1.110319,0.092200
9,Sheldon Richardson,109,1.521554,1.430948,0.090606


In [23]:
top_20_gravity_scores = (
    gravity_dataset
    .groupby("rusher_name", as_index=False)
    .agg(
        num_rows=("gravity_score", "size"),
        actual_attention=("avg_attention_score", "mean"),
        expected_attention=("expected_attention", "mean"),
        gravity_score=("gravity_score", "mean"),
    )
    .query("num_rows >= 75")
    .sort_values("gravity_score", ascending=False)
    .tail(20)
    .reset_index(drop=True)
)

top_20_gravity_scores

,rusher_name,num_rows,actual_attention,expected_attention,gravity_score
0,Sam Hubbard,198,0.947063,1.000594,-0.053531
1,Alex Okafor,103,0.937559,0.994744,-0.057185
2,Jeffery Simmons,224,1.363899,1.430387,-0.066488
3,Brandon Williams,80,1.384607,1.453509,-0.068902
4,Malcolm Roach,77,1.383572,1.453401,-0.069830
5,Daron Payne,202,1.349734,1.419806,-0.070072
6,Larry Ogunjobi,178,1.409286,1.480602,-0.071317
7,Nicholas Williams,112,1.383278,1.455491,-0.072213
8,Kerry Hyder,155,1.258736,1.334786,-0.076050
9,Mario Addison,94,0.916008,0.999500,-0.083492
